In [1]:
#load the data
import pandas as pd

df = pd.read_csv("../../data/samples/yelp_merchant_engagement.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (8812, 33)
Columns: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours', 'total_reviews', 'earlier_reviews', 'recent_reviews', 'review_growth', 'average_rating', 'earlier_average_rating', 'recent_average_rating', 'rating_change', 'total_checkins', 'earlier_checkins', 'recent_checkins', 'checkin_growth', 'review_change', 'checkin_change', 'rating_change_for_score', 'review_trend_norm', 'checkin_trend_norm', 'rating_trend_norm', 'engagement_score']


In [2]:
#examine the actual quartiles
q25 = df["engagement_score"].quantile(0.25)
q50 = df["engagement_score"].quantile(0.50)
q75 = df["engagement_score"].quantile(0.75)

print("25th percentile:", q25)
print("50th percentile:", q50)
print("75th percentile:", q75)

25th percentile: 0.5135836385836385
50th percentile: 0.5178571428571428
75th percentile: 0.5198412698412698


In [3]:
#check how many merchants exactly sit on those thresholds
print("Merchants exactly at Q25:", (df["engagement_score"] == q25).sum())
print("Merchants exactly at Q50:", (df["engagement_score"] == q50).sum())
print("Merchants exactly at Q75:", (df["engagement_score"] == q75).sum())

Merchants exactly at Q25: 277
Merchants exactly at Q50: 3819
Merchants exactly at Q75: 81


In [4]:
#create merchant status
df["merchant_status"] = pd.cut(
    df["engagement_score"],
    bins=[-float("inf"), q25, q75, float("inf")],
    labels=["Declining", "Stable", "Growing"],
    include_lowest=True
)

In [5]:
#check the classification
status_counts = df["merchant_status"].value_counts().sort_index()

print(status_counts)

print("\nPercent distribution:")
print(
    (df["merchant_status"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

merchant_status
Declining    2361
Stable       4266
Growing      2185
Name: count, dtype: int64

Percent distribution:
merchant_status
Declining    26.79
Stable       48.41
Growing      24.80
Name: proportion, dtype: float64


In [6]:
#verify there is no missing classification
print("\nMissing merchant_status:", df["merchant_status"].isna().sum())


Missing merchant_status: 0


In [7]:
print("\nThresholds:")
print("Declining <= ", q25)
print("Stable      > ", q25, "and <= ", q75)
print("Growing     > ", q75)


Thresholds:
Declining <=  0.5135836385836385
Stable      >  0.5135836385836385 and <=  0.5198412698412698
Growing     >  0.5198412698412698


In [8]:
#create the summary table
merchant_summary = df[
    [
        "name",
        "city",
        "review_change",
        "checkin_change",
        "rating_change",
        "engagement_score",
        "merchant_status"
    ]
].rename(
    columns={
        "name": "Business",
        "city": "City",
        "review_change": "Review Trend",
        "checkin_change": "Check-in Trend",
        "rating_change": "Rating Trend",
        "engagement_score": "Engagement Score",
        "merchant_status": "Status"
    }
)

print("Summary table shape:", merchant_summary.shape)

merchant_summary.head()

Summary table shape: (8812, 7)


,Business,City,Review Trend,Check-in Trend,Rating Trend,Engagement Score,Status
0,St Honore Pastries,Philadelphia,4,4,3.800000,0.648443,Growing
1,Tuna Bar,Philadelphia,16,-8,0.081699,0.572630,Growing
2,BAP,Philadelphia,5,1,-0.500000,0.527320,Growing
3,Bar One,Philadelphia,0,0,NaN,0.517857,Stable
4,DeSandro on Main,Philadelphia,0,0,NaN,0.517857,Stable


In [9]:
#show the top 5 from each status
for status in ["Growing", "Stable", "Declining"]:
    print(f"\n{'=' * 60}")
    print(f"{status} — Top 5 by Engagement Score")
    print(f"{'=' * 60}")

    display(
        merchant_summary[
            merchant_summary["Status"] == status
        ]
        .sort_values("Engagement Score", ascending=False)
        .head(5)
    )


Growing — Top 5 by Engagement Score


,Business,City,Review Trend,Check-in Trend,Rating Trend,Engagement Score,Status
8361,Reading Terminal Market,Philadelphia,50,73,-0.055996,0.874818,Growing
4764,Matcha Cafe Maiko,Philadelphia,46,70,NaN,0.853327,Growing
2207,Victory Brewing Company Philadelphia,Philadelphia,52,36,NaN,0.811508,Growing
3307,Sushi Yama,Tampa,46,44,NaN,0.801740,Growing
1262,Terakawa Ramen,Philadelphia,27,75,-0.191667,0.776727,Growing



Stable — Top 5 by Engagement Score


,Business,City,Review Trend,Check-in Trend,Rating Trend,Engagement Score,Status
1880,Balducci's,Philadelphia,0,1,NaN,0.519841,Stable
8090,Watkins Drinkery,Philadelphia,0,1,NaN,0.519841,Stable
358,Foodery,Philadelphia,0,1,NaN,0.519841,Stable
157,Three 12 Sport Bar and Lounge,Philadelphia,0,1,0.0,0.519841,Stable
4925,Uncle's Seafood,Philadelphia,0,1,NaN,0.519841,Stable



Declining — Top 5 by Engagement Score


,Business,City,Review Trend,Check-in Trend,Rating Trend,Engagement Score,Status
34,The Corner Cafe and Deli,Tampa,-1,0,NaN,0.513584,Declining
41,Checkers,Philadelphia,-1,0,NaN,0.513584,Declining
7451,Draught Horse Pub & Grill,Philadelphia,-1,0,NaN,0.513584,Declining
7422,Bellas Cafe,Tampa,-1,0,NaN,0.513584,Declining
7416,Gold Star Pizza,Philadelphia,-1,0,NaN,0.513584,Declining


#### Merchant Health Classification

Merchants were classified into **Declining, Stable, and Growing** categories based on the `engagement_score`.

**Classification Method**

Quartile-based thresholds were used to reflect the observed score distribution:
* **Declining:** `engagement_score <= 0.5135836386`
* **Stable:** `0.5135836386 < engagement_score <= 0.5198412698`
* **Growing:** `engagement_score > 0.5198412698`

Ties at the threshold values were kept together, resulting in a slightly different distribution from an exact 25% / 50% / 25% split.

**Results**

| Status    | Merchants |       Share |
| --------- | --------: | ----------: |
| Declining |     2,361 |      26.79% |
| Stable    |     4,266 |      48.41% |
| Growing   |     2,185 |      24.80% |
| **Total** | **8,812** | **100.00%** |

A merchant-level summary containing business information, trend metrics, engagement score, and health status was also created. Representative merchants from each category were reviewed as a sanity check.

**Output:** `data/samples/yelp_merchant_engagement.csv`


In [10]:
#save the updated dataset
output_path = "../../data/samples/yelp_merchant_engagement.csv"

df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print("Saved dataset shape:", df.shape)

Saved: ../../data/samples/yelp_merchant_engagement.csv
Saved dataset shape: (8812, 34)


In [11]:
#relode and validate
df_check = pd.read_csv(output_path)

# Structural validation
assert df_check.shape == df.shape
assert df_check["business_id"].is_unique
assert not df_check.duplicated().any()

# Merchant status validation
assert "merchant_status" in df_check.columns
assert df_check["merchant_status"].notna().all()

# Confirm only the expected categories exist
expected_statuses = {"Declining", "Stable", "Growing"}

assert set(df_check["merchant_status"].unique()) == expected_statuses

# Confirm all merchants are classified
assert df_check["merchant_status"].value_counts().sum() == 8812

print("Final validation passed.")
print("Shape:", df_check.shape)
print("Unique businesses:", df_check["business_id"].nunique())
print("Duplicate rows:", df_check.duplicated().sum())
print("Missing merchant_status:", df_check["merchant_status"].isna().sum())

print("\nMerchant status counts:")
print(df_check["merchant_status"].value_counts().sort_index())

Final validation passed.
Shape: (8812, 34)
Unique businesses: 8812
Duplicate rows: 0
Missing merchant_status: 0

Merchant status counts:
merchant_status
Declining    2361
Growing      2185
Stable       4266
Name: count, dtype: int64
